# CRNN + CTC baseline training in Google Colab

Clean submission workflow for the `cyrillic-htr` project.

This notebook is intended for reproducible baseline training in Google Colab with:

- Google Drive mounted for persistent checkpoints, logs, and plots;
- DVC remotes configured on Google Drive;
- MLflow server running locally inside the Colab runtime;
- resumable training from `last.ckpt`;
- final trained checkpoints pushed to the DVC `models_remote`.

## 1. Check GPU and mount Google Drive

In [1]:
import torch
from google.colab import drive

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Mounted at /content/drive


## 2. Project paths

In [2]:
REPO_URL = "https://github.com/elizaveta112/cyrillic-htr.git"
REPO_DIR = "/content/cyrillic-htr"

DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/cyrillic_htr"
DRIVE_CHECKPOINT_DIR = f"{DRIVE_PROJECT_ROOT}/checkpoints/crnn_ctc"
DRIVE_LOG_DIR = f"{DRIVE_PROJECT_ROOT}/logs"
DRIVE_PLOTS_DIR = f"{DRIVE_PROJECT_ROOT}/plots"

DRIVE_DVC_ROOT = "/content/drive/MyDrive/cyrillic_htr_dvc"
DRIVE_DVC_DATA = f"{DRIVE_DVC_ROOT}/data"
DRIVE_DVC_MODELS = f"{DRIVE_DVC_ROOT}/models"

KAGGLE_JSON_PATH = "/content/drive/MyDrive/kaggle/kaggle.json"

print("Repo:", REPO_URL)
print("Repo dir:", REPO_DIR)
print("Checkpoints:", DRIVE_CHECKPOINT_DIR)
print("Logs:", DRIVE_LOG_DIR)
print("Plots:", DRIVE_PLOTS_DIR)
print("DVC data remote:", DRIVE_DVC_DATA)
print("DVC models remote:", DRIVE_DVC_MODELS)

Repo: https://github.com/elizaveta112/cyrillic-htr.git
Repo dir: /content/cyrillic-htr
Checkpoints: /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc
Logs: /content/drive/MyDrive/cyrillic_htr/logs
Plots: /content/drive/MyDrive/cyrillic_htr/plots
DVC data remote: /content/drive/MyDrive/cyrillic_htr_dvc/data
DVC models remote: /content/drive/MyDrive/cyrillic_htr_dvc/models


## 3. Clone or update repository

In [3]:
%%bash
set -e

if [ -d "/content/cyrillic-htr/.git" ]; then
  cd /content/cyrillic-htr
  git pull
else
  git clone https://github.com/elizaveta112/cyrillic-htr.git /content/cyrillic-htr
  cd /content/cyrillic-htr
fi

pwd
git status

/content/cyrillic-htr
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Cloning into '/content/cyrillic-htr'...


## 4. Install dependencies


In [5]:
%%bash
set -e

cd /content/cyrillic-htr

pip install -q uv

uv sync --frozen || uv sync

uv run python --version
uv run python -c "import torch; print('torch:', torch.__version__)"
uv run python -c "import pandas; import mlflow; print('pandas:', pandas.__version__, 'mlflow:', mlflow.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 29.1 MB/s eta 0:00:00
Python 3.11.15
torch: 2.12.0+cu130
pandas: 2.3.3 mlflow: 3.12.0


Using CPython 3.11.15
Creating virtual environment at: .venv
Prepared 215 packages in 1m 03s
Installed 215 packages in 1.17s
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.13.5
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0
 + alembic==1.18.4
 + amqp==5.3.1
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.13.0
 + appdirs==1.4.4
 + asyncssh==2.23.0
 + atpublic==7.0.0
 + attrs==26.1.0
 + billiard==4.2.4
 + bleach==6.3.0
 + blinker==1.9.0
 + cachetools==7.1.4
 + celery==5.6.3
 + certifi==2026.5.20
 + cffi==2.0.0
 + cfgv==3.5.0
 + charset-normalizer==3.4.7
 + click==8.4.0
 + click-didyoumean==0.3.1
 + click-plugins==1.1.1.2
 + click-repl==0.3.0
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + configobj==5.0.9
 + contourpy==1.3.3
 + cryptography==46.0.7
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + cyrillic-htr==0.1.0 (from file:///content/cyrillic-htr)
 + databricks-sdk==0.111.0
 + dictdiffer==0.9.0
 + diskca

## 5. Kaggle credentials

In [6]:
%%bash
set -e

cd /content/cyrillic-htr

mkdir -p ~/.kaggle

if [ -f "/content/drive/MyDrive/kaggle/kaggle.json" ]; then
  cp /content/drive/MyDrive/kaggle/kaggle.json ~/.kaggle/kaggle.json
  chmod 600 ~/.kaggle/kaggle.json
  uv run kaggle datasets list -s "cyrillic handwriting" | head -20
else
  echo "kaggle.json not found at /content/drive/MyDrive/kaggle/kaggle.json"
  echo "This is okay if data already exists in DVC remote and you only use dvc pull."
fi

ref                                               title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
constantinwerner/cyrillic-handwriting-dataset     Cyrillic Handwriting Dataset                        1623532905  2025-10-22 13:36:21.893000           6115         86                1  
mariuspenteliuc/rts-ocr                           19th-Century Romanian Transitional Script            267843139  2024-05-21 15:54:03.860000            273         28                1  
nenadlukic/openwrite-dataset                      OpenWrite Dataset                                    917643825  2025-09-16 11:46:52.733000             35          4           0.9375  
vladimirmynka/aug-cyrillic-handwriting-dataset    Aug Cyrillic Handwri

## 6. Configure DVC remotes on Google Drive

In [7]:
%%bash
set -e

cd /content/cyrillic-htr

rm -f .dvc/config.local

mkdir -p /content/drive/MyDrive/cyrillic_htr_dvc/data
mkdir -p /content/drive/MyDrive/cyrillic_htr_dvc/models

uv run dvc remote modify --local data_remote url /content/drive/MyDrive/cyrillic_htr_dvc/data
uv run dvc remote modify --local models_remote url /content/drive/MyDrive/cyrillic_htr_dvc/models

uv run dvc remote list

data_remote     /content/drive/MyDrive/cyrillic_htr_dvc/data    (default)
models_remote   /content/drive/MyDrive/cyrillic_htr_dvc/models


## 7. Repair incomplete DVC data remote

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

echo "Cleaning incomplete Drive DVC data remote..."
rm -rf /content/drive/MyDrive/cyrillic_htr_dvc/data/*
mkdir -p /content/drive/MyDrive/cyrillic_htr_dvc/data

echo "Cleaning local data workspace..."
rm -rf data/raw/cyrillic-handwriting-dataset
rm -rf data/processed

echo "Downloading Kaggle dataset..."
uv run python scripts/download_data.py data.force_download=true

echo "Checking raw dataset..."
du -sh data/raw/cyrillic-handwriting-dataset
find data/raw/cyrillic-handwriting-dataset -type f | wc -l
ls -lh data/raw/cyrillic-handwriting-dataset | head

echo "Preparing splits and vocab..."
uv run python scripts/prepare_splits.py
uv run python scripts/preprocess_data.py

echo "Checking processed data..."
ls -lh data/processed
head data/processed/train_split.tsv
head data/processed/vocab.json

echo "Recreating DVC metadata..."
uv run dvc add data/raw/cyrillic-handwriting-dataset
uv run dvc add data/processed

echo "Pushing full data cache to Google Drive DVC remote..."
uv run dvc push -r data_remote data/raw/cyrillic-handwriting-dataset.dvc data/processed.dvc

echo "Drive DVC remote size after push:"
du -sh /content/drive/MyDrive/cyrillic_htr_dvc/data

echo "Drive DVC remote file count:"
find /content/drive/MyDrive/cyrillic_htr_dvc/data -type f | wc -l

## 8. `dvc pull`



In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

echo "Drive DVC data remote size:"
du -sh /content/drive/MyDrive/cyrillic_htr_dvc/data || true

echo "Pulling data from DVC remote..."
uv run dvc pull -j 1 -r data_remote   data/raw/cyrillic-handwriting-dataset.dvc   data/processed.dvc

echo "Checking restored data:"
du -sh data/raw/cyrillic-handwriting-dataset
du -sh data/processed
ls -lh data/processed

Process is terminated.


Optional fallback: download data from Kaggle

In [8]:
%%bash
set -e

cd /content/cyrillic-htr

if [ -d "data/raw/cyrillic-handwriting-dataset" ] && [ -d "data/processed" ]; then
  echo "Data already exists locally. Skipping data preparation."
else
  echo "Data not found locally. Downloading from Kaggle instead of slow DVC pull..."
  uv run python scripts/download_data.py data.force_download=true
  uv run python scripts/prepare_splits.py
  uv run python scripts/preprocess_data.py
fi

du -sh data/raw/cyrillic-handwriting-dataset
du -sh data/processed
ls -lh data/processed

Data not found locally. Downloading from Kaggle instead of slow DVC pull...
Dataset URL: https://www.kaggle.com/datasets/constantinwerner/cyrillic-handwriting-dataset
License(s): CC0-1.0

Dataset downloaded to: data/raw/cyrillic-handwriting-dataset
Train split saved to: data/processed/train_split.tsv
Validation split saved to: data/processed/val_split.tsv
Test split saved to: data/processed/test_split.tsv
Train size: 65057
Validation size: 7229
Test size: 1544
Dropped rows with empty target text: 2
Vocabulary saved to: data/processed/vocab.json
Vocabulary size: 106
1.7G	data/raw/cyrillic-handwriting-dataset
2.0M	data/processed
total 2.0M
-rw-r--r-- 1 root root  48K Jun  2 05:48 test_split.tsv
-rw-r--r-- 1 root root 1.7M Jun  2 05:48 train_split.tsv
-rw-r--r-- 1 root root 194K Jun  2 05:48 val_split.tsv
-rw-r--r-- 1 root root 1.3K Jun  2 05:48 vocab.json


100%|██████████| 1.51G/1.51G [00:25<00:00, 64.9MB/s]


## 8. Validate vocabulary coverage

In [10]:
%%bash
set -e

cd /content/cyrillic-htr

uv run python - <<'PY'
import json
from pathlib import Path

import pandas as pd

vocab = json.loads(Path("data/processed/vocab.json").read_text(encoding="utf-8"))
allowed = set(vocab)

has_unknown = False

for split_name in ["train_split", "val_split", "test_split"]:
    path = Path(f"data/processed/{split_name}.tsv")
    df = pd.read_csv(path, sep="\t")

    unknown = set()
    for text in df["text"].dropna().astype(str):
        unknown.update(character for character in text if character not in allowed)

    print(split_name, "unknown:", sorted(unknown))
    has_unknown = has_unknown or bool(unknown)

if has_unknown:
    raise RuntimeError("Unknown characters were found in split files.")
PY

train_split unknown: []
val_split unknown: []
test_split unknown: []


## 9. Create persistent Drive directories

In [13]:
%%bash
set -e

mkdir -p /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc
mkdir -p /content/drive/MyDrive/cyrillic_htr/logs
mkdir -p /content/drive/MyDrive/cyrillic_htr/plots

echo "Existing checkpoints:"
find /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc -maxdepth 1 -type f || true

Existing checkpoints:
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/best-epoch=32-val_cer=0.0832.ckpt
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/last.ckpt


## 10. Start MLflow server

In [14]:
%%bash
set -e

cd /content/cyrillic-htr

pkill -f "mlflow server" || true
sleep 3

rm -f mlflow.log

nohup uv run mlflow server \
  --host 127.0.0.1 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns > mlflow.log 2>&1 &

echo "MLflow server is starting..."
sleep 10
tail -n 30 mlflow.log || true

MLflow server is starting...
Registry store URI not provided. Using backend store URI.
2026/06/02 05:50:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/02 05:50:19 INFO mlflow.store.db.utils: Updating database tables
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.


## 11. Wait until MLflow is ready

In [15]:
%%bash
set -e

cd /content/cyrillic-htr

uv run python - <<'PY'
import time
from mlflow.tracking import MlflowClient

tracking_uri = "http://127.0.0.1:5000"

for attempt in range(18):
    try:
        client = MlflowClient(tracking_uri=tracking_uri)
        experiments = client.search_experiments()
        print("MLflow is ready")
        print("Experiments:", [experiment.name for experiment in experiments])
        break
    except Exception as error:
        print(f"Attempt {attempt + 1}/18: MLflow is not ready yet: {error}")
        time.sleep(10)
else:
    print("Last MLflow log lines:")
    print(open("mlflow.log", encoding="utf-8").read()[-4000:])
    raise RuntimeError("MLflow did not become ready")
PY

MLflow is ready
Experiments: ['Default']


In [16]:
%%bash
ps aux | grep "mlflow server" | grep -v grep || true

root        3205  0.0  0.3 316760 40180 ?        Sl   05:50   0:00 uv run mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns
root        3214  9.6  1.8 674564 247920 ?       Sl   05:50   0:05 /content/cyrillic-htr/.venv/bin/python /content/cyrillic-htr/.venv/bin/mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns


## 12. Full training with MLflow and auto-resume

This cell can be rerun after a Colab disconnect.

- `train.save_dir` points to Google Drive.
- `last.ckpt` is used for resume.
- `train.epochs` is the final target epoch count.


In [17]:
%%bash
set -e

cd /content/cyrillic-htr

export MPLBACKEND=Agg

uv run python scripts/train.py \
  model=crnn_ctc \
  train=colab \
  train.epochs=50 \
  dvc.enabled=false \
  logger.tracking_uri=http://127.0.0.1:5000 \
  train.save_dir=/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc \
  train.log_dir=/content/drive/MyDrive/cyrillic_htr/logs \
  train.plots_dir=/content/drive/MyDrive/cyrillic_htr/plots \
  +train.resume_from_checkpoint=auto

Resuming training from checkpoint: /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/last.ckpt
┏━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model    │ CRNNCTC │  7.7 M │ train │     0 │
│ 1 │ ctc_loss │ CTCLoss │      0 │ train │     0 │
└───┴──────────┴─────────┴────────┴───────┴───────┘
Trainable params: 7.7 M                                                         
Non-trainable params: 0                                                         
Total params: 7.7 M                                                             
Total estimated model params size (MB): 30                                      
Modules in train mode: 27                                                       
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
Epoch 49/49 ━━━━━━━

Experiment with name cyrillic-htr not found. Creating it.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
/content/cyrillic-htr/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc exists and is not empty.
Restoring states from the checkpoint path at /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/la

## 13. Inspect checkpoints, plots, CSV logs, and MLflow artifacts

In [18]:
%%bash
set -e

cd /content/cyrillic-htr

echo "Checkpoints on Drive:"
find /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc -maxdepth 1 -type f || true

echo ""
echo "Plots on Drive:"
find /content/drive/MyDrive/cyrillic_htr/plots -maxdepth 1 -type f || true

echo ""
echo "CSV logs on Drive:"
find /content/drive/MyDrive/cyrillic_htr/logs -path "*metrics.csv" -type f || true

echo ""
echo "MLflow local files:"
find mlruns -maxdepth 3 -type f | head -50 || true

Checkpoints on Drive:
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/last.ckpt
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/best-epoch=36-val_cer=0.0823.ckpt

Plots on Drive:
/content/drive/MyDrive/cyrillic_htr/plots/test_edit_similarity.png
/content/drive/MyDrive/cyrillic_htr/plots/test_cer.png
/content/drive/MyDrive/cyrillic_htr/plots/val_loss.png
/content/drive/MyDrive/cyrillic_htr/plots/val_valid_character_rate.png
/content/drive/MyDrive/cyrillic_htr/plots/val_wer.png
/content/drive/MyDrive/cyrillic_htr/plots/test_wer.png
/content/drive/MyDrive/cyrillic_htr/plots/test_valid_character_rate.png
/content/drive/MyDrive/cyrillic_htr/plots/test_line_accuracy.png
/content/drive/MyDrive/cyrillic_htr/plots/test_loss.png
/content/drive/MyDrive/cyrillic_htr/plots/val_edit_similarity.png
/content/drive/MyDrive/cyrillic_htr/plots/val_line_accuracy.png
/content/drive/MyDrive/cyrillic_htr/plots/train_loss_step.png
/content/drive/MyDrive/cyrillic_htr/plots/val_cer.png
/conten

## 14. Push trained checkpoints to DVC models remote


In [20]:
%%bash
set -e

cd /content/cyrillic-htr

echo "Files tracked by Git inside models:"
git ls-files models || true

Files tracked by Git inside models:
models/.gitkeep


In [21]:
%%bash
set -e

cd /content/cyrillic-htr

git rm -r --cached models || true
rm -f models/.gitkeep

mkdir -p models/crnn_ctc
cp -f /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/*.ckpt models/crnn_ctc/

uv run dvc add models
uv run dvc push -j 1 -r models_remote models.dvc

echo "Models remote size:"
du -sh /content/drive/MyDrive/cyrillic_htr_dvc/models

echo ""
echo "DVC status:"
uv run dvc status

echo ""
echo "Git status:"
git status --short

rm 'models/.gitkeep'

To track the changes with git, run:

	git add models.dvc .gitignore

To enable auto staging, run:

	dvc config core.autostage true
2 files pushed
Models remote size:
89M	/content/drive/MyDrive/cyrillic_htr_dvc/models

DVC status:
data/raw/cyrillic-handwriting-dataset.dvc:
	changed outs:
		not in cache:       data/raw/cyrillic-handwriting-dataset

Git status:
 M .gitignore
 M data/processed.dvc
D  models/.gitkeep
 M src/cyrillic_htr/training/runner.py
?? mlflow.log
?? models.dvc


## 15. Final check

In [22]:
%%bash
set -e

cd /content/cyrillic-htr

echo "DVC remotes:"
uv run dvc remote list

echo ""
echo "Raw data:"
du -sh data/raw/cyrillic-handwriting-dataset || true

echo ""
echo "Processed data:"
du -sh data/processed || true

echo ""
echo "Repo model artifacts:"
find models -maxdepth 3 -type f | head -20 || true

echo ""
echo "Drive checkpoints:"
find /content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc -maxdepth 1 -type f || true

echo ""
echo "Git status:"
git status --short

DVC remotes:
data_remote     /content/drive/MyDrive/cyrillic_htr_dvc/data    (default)
models_remote   /content/drive/MyDrive/cyrillic_htr_dvc/models

Raw data:
1.7G	data/raw/cyrillic-handwriting-dataset

Processed data:
2.0M	data/processed

Repo model artifacts:
models/crnn_ctc/best-epoch=36-val_cer=0.0823.ckpt
models/crnn_ctc/last.ckpt

Drive checkpoints:
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/last.ckpt
/content/drive/MyDrive/cyrillic_htr/checkpoints/crnn_ctc/best-epoch=36-val_cer=0.0823.ckpt

Git status:
 M .gitignore
 M data/processed.dvc
D  models/.gitkeep
 M src/cyrillic_htr/training/runner.py
?? mlflow.log
?? models.dvc


In [10]:
%%bash
set -e

cd /content/cyrillic-htr

cp models.dvc /content/drive/MyDrive/cyrillic_htr/models.dvc
cp data/processed.dvc /content/drive/MyDrive/cyrillic_htr/processed.dvc

ls -lh /content/drive/MyDrive/cyrillic_htr/models.dvc
ls -lh /content/drive/MyDrive/cyrillic_htr/processed.dvc

-rw------- 1 root root 107 Jun  3 10:24 /content/drive/MyDrive/cyrillic_htr/models.dvc
-rw------- 1 root root 108 Jun  3 10:24 /content/drive/MyDrive/cyrillic_htr/processed.dvc
